# 01 — Análise Exploratória de Dados (EDA)

Este notebook realiza a análise exploratória completa do dataset de voos, cobrindo:

- Estatísticas descritivas
- Tratamento de valores ausentes
- Visualização de distribuições e padrões
- Geração de insights para a modelagem


In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_or_generate
from visualization import set_style, plot_delay_distribution, plot_delay_by_category, plot_correlation_matrix

set_style()

DATA_PATH = '../data/voos.csv'
df = load_or_generate(DATA_PATH, n_samples=10_000)
print(f"Dataset carregado: {df.shape[0]:,} voos x {df.shape[1]} colunas")
df.head()


## 1. Visão Geral dos Dados

In [ ]:
print("Dimensões:", df.shape)
print()
print("Tipos de dados:")
print(df.dtypes)


In [ ]:
df.describe(include='all').T


## 2. Análise de Valores Ausentes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Ausentes': missing, '% do Total': missing_pct})
missing_df = missing_df[missing_df['Ausentes'] > 0].sort_values('% do Total', ascending=False)
print("Colunas com valores ausentes:")
display(missing_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing_df['% do Total'].plot(kind='barh', ax=ax, color='#E53935')
ax.set_title('Percentual de Valores Ausentes por Coluna')
ax.set_xlabel('% Ausente')
plt.tight_layout()
plt.show()
print("\nEstratégia adotada: preenchimento com mediana (numérico) e moda (categórico).")


## 3. Distribuição dos Atrasos

In [ ]:
fig = plot_delay_distribution(df, 'arrival_delay_min')
plt.show()

fig = plot_delay_distribution(df, 'departure_delay_min')
plt.show()

delayed_pct = df['is_delayed'].mean() * 100
print(f"\nVoos com atraso na chegada (≥15 min): {delayed_pct:.1f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(
    [df.loc[df['is_delayed']==0, 'arrival_delay_min'].dropna(),
     df.loc[df['is_delayed']==1, 'arrival_delay_min'].dropna()],
    labels=['Não Atrasado', 'Atrasado']
)
axes[0].set_title('Atraso de Chegada por Status')
axes[0].set_ylabel('Minutos')

df['departure_delay_min'].dropna().hist(bins=50, ax=axes[1], color='#1E88E5', edgecolor='white')
axes[1].set_title('Distribuição do Atraso na Partida')
axes[1].set_xlabel('Minutos')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()


## 4. Análise por Variáveis Categóricas

In [ ]:
for col in ['carrier', 'weather', 'origin', 'destination']:
    fig = plot_delay_by_category(df, col)
    plt.show()


## 5. Padrões Temporais

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

delay_by_month = df.groupby('month')['is_delayed'].mean()
axes[0].bar(delay_by_month.index, delay_by_month.values, color='#42A5F5')
axes[0].set_title('Taxa de Atraso por Mês')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel('Taxa de Atraso')
axes[0].set_xticks(range(1, 13))

delay_by_dow = df.groupby('day_of_week')['is_delayed'].mean()
day_labels = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
axes[1].bar(range(1, 8), delay_by_dow.values, color='#42A5F5')
axes[1].set_title('Taxa de Atraso por Dia da Semana')
axes[1].set_xlabel('Dia da Semana')
axes[1].set_ylabel('Taxa de Atraso')
axes[1].set_xticks(range(1, 8))
axes[1].set_xticklabels(day_labels)

plt.tight_layout()
plt.show()


In [ ]:
delay_by_hour = df.groupby('hour')['is_delayed'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(delay_by_hour.index, delay_by_hour.values, marker='o', color='#1565C0')
ax.fill_between(delay_by_hour.index, delay_by_hour.values, alpha=0.3, color='#42A5F5')
ax.set_title('Taxa de Atraso por Hora do Dia')
ax.set_xlabel('Hora')
ax.set_ylabel('Taxa de Atraso')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()


## 6. Correlação entre Variáveis Numéricas

In [ ]:
num_cols = ['month', 'day_of_week', 'hour', 'distance_km',
            'departure_delay_min', 'arrival_delay_min', 'is_delayed']
fig = plot_correlation_matrix(df, num_cols)
plt.show()


## 7. Relação Distância × Atraso

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sample = df.dropna(subset=['distance_km', 'arrival_delay_min']).sample(2000, random_state=42)
ax.scatter(sample['distance_km'], sample['arrival_delay_min'],
           c=sample['is_delayed'], cmap='RdYlGn_r', alpha=0.4, s=15)
ax.set_title('Distância vs Atraso de Chegada')
ax.set_xlabel('Distância (km)')
ax.set_ylabel('Atraso de Chegada (min)')
plt.colorbar(ax.collections[0], ax=ax, label='Atrasado')
plt.tight_layout()
plt.show()


## 8. Insights Principais

Com base na análise exploratória, destacam-se os seguintes insights:

1. **Taxa de atraso global ~35%**: Aproximadamente um em cada três voos chega com atraso ≥ 15 min, indicando um problema relevante nas operações.

2. **Clima é o fator mais impactante**: Voos em condições de tempestade (`storm`) apresentam taxa de atraso muito superior à média, seguidos de chuva (`rain`). Clima limpo tem a menor taxa.

3. **Horário de pico vespertino**: Voos com partida após as 17h apresentam taxas de atraso sistematicamente mais altas, sugerindo efeito cascata das operações ao longo do dia.

4. **Forte correlação atraso de partida → chegada**: A correlação entre `departure_delay_min` e `arrival_delay_min` é alta (>0.9), tornando o atraso de partida a feature mais preditiva para o modelo supervisionado.

5. **Finais de semana com mais atrasos**: Sábado e domingo apresentam taxa de atraso ligeiramente superior aos dias úteis.

6. **Distância tem baixa influência direta**: A correlação entre distância e atraso é pequena, indicando que voos longos não são intrinsecamente mais propensos a atrasos — o fator determinante é operacional.
